In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Preprocesamiento
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Selección de características
from sklearn.feature_selection import SequentialFeatureSelector

# Reducción de dimensionalidad
from sklearn.decomposition import PCA

# Model selection
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV

# Métricas
from sklearn.metrics import accuracy_score, classification_report, recall_score, roc_auc_score, roc_curve, auc, confusion_matrix, f1_score

# Pipelines y balanceo
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE


In [10]:
churn_train = pd.read_csv('churn_train_clean.csv')
churn_test = pd.read_csv('churn_test_clean.csv')
X_train = churn_train.drop(columns='Churn Value')
Y_train = churn_train['Churn Value']
X_test = churn_test.drop(columns='Churn Value')
Y_test = churn_test['Churn Value']

In [11]:
numerical_cols = ["Total Charges", "Monthly Charges", "Tenure Months", "CLTV"]
categorical_cols = [
    "Gender", "Senior Citizen", "Partner", "Dependents",
    "Phone Service", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method"
]

# RANDOM FOREST

In [18]:
# Preprocesador para las transformaciones de columnas

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])


# Pipeline con el preprocesador, el balanceo y el modelo

pipeline_rf = Pipeline([
    ("preprocessor", preprocessor),         # transformar datos
    ("oversample", RandomOverSampler(random_state=10)),  # balancear clases
    ("rf", RandomForestClassifier(random_state=10))
])


# Parámetros a probar por el modelo
param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [2, 4, 6, 8]
    }

#Grid search
grid_search = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid,
    scoring="f1",
    cv=4,
    verbose=1,
    n_jobs=-1
)
print("Ejecutando GridSearchCV...")
grid_search.fit(X_train, Y_train)
print("GridSearch completo.\n")

print("Mejores parámetros encontrados:")
print(grid_search.best_params_)

print(f"\nMejor score CV (F1): {grid_search.best_score_:.4f}")


#Evaluación final en test
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n=== REPORTE FINAL EN TEST ===")
print(classification_report(Y_test, y_pred))

final_recall = recall_score(Y_test, y_pred)
print(f"Recall Final en TEST: {final_recall:.4f}")


Ejecutando GridSearchCV...
Fitting 4 folds for each of 12 candidates, totalling 48 fits
GridSearch completo.

Mejores parámetros encontrados:
{'rf__max_depth': 8, 'rf__n_estimators': 300}

Mejor score CV (F1): 0.6440

=== REPORTE FINAL EN TEST ===
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      1294
           1       0.54      0.80      0.64       467

    accuracy                           0.76      1761
   macro avg       0.72      0.77      0.73      1761
weighted avg       0.81      0.76      0.77      1761

Recall Final en TEST: 0.7987


# XGBOOST

In [19]:
# Preprocesador para las transformaciones de columnas

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])


# Pipeline con el preprocesador, el balanceo y el modelo

pipeline_xgb = Pipeline([
    ("preprocessor", preprocessor),        
    ("oversample", RandomOverSampler(random_state=10)),  
    ("xgb", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=10,
        n_jobs=-1
    ))
])

# Parámetros a probar por el modelo
param_grid = {
    "xgb__n_estimators": [100, 200, 300],
    "xgb__max_depth": [2, 4, 6, 8],
    "xgb__learning_rate": [0.05, 0.1, 0.2],

}


#Grid search

grid_search = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid,
    scoring="recall",
    cv=4,
    verbose=1,
    n_jobs=-1
)

print("Ejecutando GridSearchCV...")
grid_search.fit(X_train, Y_train)
print("GridSearch completo.\n")

print("Mejores parámetros encontrados:")
print(grid_search.best_params_)

print(f"\nMejor score CV (Recall): {grid_search.best_score_:.4f}")



#Evaluación final en test

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n=== REPORTE FINAL EN TEST ===")
print(classification_report(Y_test, y_pred))

final_recall = recall_score(Y_test, y_pred)
print(f"Recall Final en TEST: {final_recall:.4f}")


Ejecutando GridSearchCV...
Fitting 4 folds for each of 36 candidates, totalling 144 fits
GridSearch completo.

Mejores parámetros encontrados:
{'xgb__learning_rate': 0.05, 'xgb__max_depth': 2, 'xgb__n_estimators': 100}

Mejor score CV (Recall): 0.8096

=== REPORTE FINAL EN TEST ===
              precision    recall  f1-score   support

           0       0.92      0.72      0.81      1294
           1       0.52      0.83      0.64       467

    accuracy                           0.75      1761
   macro avg       0.72      0.77      0.72      1761
weighted avg       0.81      0.75      0.76      1761

Recall Final en TEST: 0.8266


# Naive Bayes Gaussiano

In [21]:
# Procesador de columnas

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

# Pipeline con preprocesador, balanceo y modelo

pipeline_nb = Pipeline([
    ("preprocessor", preprocessor),
    ("undersampler", RandomOverSampler(random_state=10)),
    ("nb", GaussianNB())
])


print("Ejecutando Cross-Validation (CV=5)...")
cv_scores = cross_val_score(
    pipeline_nb, 
    X_train,      # aquí usamos directamente X_train
    Y_train,
    cv=5, 
    scoring='f1', 
    n_jobs=-1
)

print(f"F1 promedio en CV: {cv_scores.mean():.4f}")
print(f"Desviación estándar en CV: {cv_scores.std():.4f}")

# ============================================================
# Entrenamiento final sobre todo el train
# ============================================================

print("\nEntrenando GaussianNB con oversampling sobre todo el train...")
pipeline_nb.fit(X_train, Y_train)
print("Entrenamiento completado.")

# ============================================================
# Predicción en test
# ============================================================

y_pred = pipeline_nb.predict(X_test)

print("\n=== REPORTE FINAL ===")
print(classification_report(Y_test, y_pred))

final_recall = recall_score(Y_test, y_pred)
print(f"Recall Final en TEST: {final_recall:.4f}")


Ejecutando Cross-Validation (CV=5)...
F1 promedio en CV: 0.6262
Desviación estándar en CV: 0.0164

Entrenando GaussianNB con oversampling sobre todo el train...
Entrenamiento completado.

=== REPORTE FINAL ===
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1294
           1       0.52      0.79      0.63       467

    accuracy                           0.75      1761
   macro avg       0.71      0.76      0.72      1761
weighted avg       0.80      0.75      0.76      1761

Recall Final en TEST: 0.7944


# Regresión logística

In [22]:

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])


pipeline_nb = Pipeline([
    ("preprocessor", preprocessor),
    ("undersampler", RandomOverSampler(random_state=10)),
    ("nb", LogisticRegression(random_state=10))
])


print("Ejecutando Cross-Validation (CV=5)...")
cv_scores = cross_val_score(
    pipeline_nb, 
    X_train,     
    Y_train,
    cv=5, 
    scoring='f1', 
    n_jobs=-1
)

print(f"F1 promedio en CV: {cv_scores.mean():.4f}")
print(f"Desviación estándar en CV: {cv_scores.std():.4f}")

# ============================================================
# Entrenamiento final sobre todo el train
# ============================================================

print("\nEntrenando regrersion logistica con oversampling sobre todo el train...")
pipeline_nb.fit(X_train, Y_train)
print("Entrenamiento completado.")

# ============================================================
# Predicción en test
# ============================================================

y_pred = pipeline_nb.predict(X_test)

print("\n=== REPORTE FINAL ===")
print(classification_report(Y_test, y_pred))

final_recall = recall_score(Y_test, y_pred)
print(f"Recall Final en TEST: {final_recall:.4f}")


Ejecutando Cross-Validation (CV=5)...
F1 promedio en CV: 0.6406
Desviación estándar en CV: 0.0129

Entrenando regrersion logistica con oversampling sobre todo el train...
Entrenamiento completado.

=== REPORTE FINAL ===
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1294
           1       0.52      0.80      0.63       467

    accuracy                           0.75      1761
   macro avg       0.71      0.76      0.72      1761
weighted avg       0.81      0.75      0.76      1761

Recall Final en TEST: 0.7966
